In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

In [46]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df[['Serangan_Jantung']]

In [47]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [48]:
log_reg = LogisticRegression()
log_reg.fit(X_train,y_train.values.ravel())

y_pred_test= log_reg.predict(X_test)
y_prob_test= log_reg.predict_proba(X_test)

In [49]:
def evaluate_model(y_pred,y_test,y_prob,evaluate,model_name='Logistic Regression'):
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)
    roc_auc = roc_auc_score(y_test,y_prob[:,1])
    logloss = log_loss(y_test,y_prob)
    
    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [55]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    acc_train = df_eval_train['Accuracy'].values[0]
    acc_test = df_eval_test['Accuracy'].values[0]
    gap = acc_train - acc_test

    if acc_train < 0.60 and acc_test < 0.60:
        status = 'Underfitting (akurasi rendah)'
    elif gap > 0.05 :
        status = f'Overfitting (gap:{gap:.2f%})'
    elif gap < -0.05:
        status = 'Overfitting / Data leakage (Test > Train)'
    else:
        status = 'Good fit'
    df_combined['Status'] = status
    return df_combined

In [56]:
df_eval_train = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate= 'Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate= 'Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)
df_eval


,Model,Evaluated,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Log Loss,Status
0,Logistic Regression,Training,0.743333,0.780899,0.785311,0.783099,0.835791,0.492335,Good fit
1,Logistic Regression,Test,0.743333,0.780899,0.785311,0.783099,0.835791,0.492335,Good fit
